In [ ]:
# @title Setup
from google.cloud import bigquery
from google.colab import data_table
import bigframes.pandas as bpd

project = 'smiling-rhythm-486213-b9'
location = 'US'
client = bigquery.Client(project=project, location=location)
data_table.enable_dataframe_formatter()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Extract the current year
from datetime import datetime
current_year = datetime.now().year
current_year

In [ ]:
# Function to execute a BigQuery query and return a DataFrame

def query_to_dataframe(query: str) -> pd.DataFrame:
    """
    Executes a SQL query in BigQuery and returns a Pandas DataFrame.

    Parameters:
    - query (str): The SQL query to execute.

    Return:
    - pd.DataFrame : The DataFrame containing the results of the query.
    """
    try:
        df = client.query(query).to_dataframe()
        print(f"Query executed successfully. Retrieved {df.shape[0]} rows.")
        return df
    except Exception as e:
        print(f"Error executing query: {e}")
        return pd.DataFrame()

## III/ Financial & Pricing Analysis

### Question 7: How does the total fare revenue of yellow taxis change over time?


In [ ]:
query_total_fare_revenue_over_time = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.total_fare_revenue_over_time`
"""
total_fare_revenue_over_time_df = query_to_dataframe(query_total_fare_revenue_over_time)
total_fare_revenue_over_time_df.head()

In [ ]:
# Filter rows where the year is between 2020 and the current year (inclusive)
filtered_total_fare_revenue_over_time_df = total_fare_revenue_over_time_df[(total_fare_revenue_over_time_df['year'] >= 2020) & (total_fare_revenue_over_time_df['year'] <= current_year)]
filtered_total_fare_revenue_over_time_df.info()

In [ ]:
# Create an area chart
fig = px.area(
    filtered_total_fare_revenue_over_time_df,
    x="trip_date",
    y=["fare_revenue", "tip_revenue", "tolls_revenue", "congestion_revenue"],
    labels={"trip_date": "Date", "value": "Revenue ($)"},
    title="NYC Taxi Revenue Trends Over Time",
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"],
)

fig.show()

In [ ]:
# Aggregate revenue by month
monthly_revenue = filtered_total_fare_revenue_over_time_df.groupby(["year", "month"])[
    ["fare_revenue", "tip_revenue", "tolls_revenue", "congestion_revenue"]
].sum().reset_index()

monthly_revenue["year_month"] = monthly_revenue["year"].astype(str) + "-" + monthly_revenue["month"].astype(str).str.zfill(2)

fig = px.bar(
    monthly_revenue,
    x="year_month",
    y=["fare_revenue", "tip_revenue", "tolls_revenue", "congestion_revenue"],
    title="Monthly Revenue Breakdown",
    labels={"value": "Revenue ($)", "year_month": "Year-Month"},
    color_discrete_sequence=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"],
)

fig.show()

In [ ]:
import plotly.io as pio

pio.renderers.default = "colab"

df = filtered_total_fare_revenue_over_time_df.copy()

if "is_weekend" not in df.columns:
    df["is_weekend"] = df["weekday"].isin([5, 6])

time_granularity_options = ["day", "week", "month", "year"]
weekend_filter_options = [("All", None), ("Weekends", True), ("Weekdays", False)]

def generate_plot(granularity, weekend_status):
    agg_column = "trip_date"
    if granularity == "week":
        agg_column = "week"
    elif granularity == "month":
        agg_column = "month"
    elif granularity == "year":
        agg_column = "year"

    filtered_df = df.copy()
    if weekend_status is not None:
        filtered_df = filtered_df[filtered_df["is_weekend"] == weekend_status]

    revenue_df = filtered_df.groupby(agg_column)["total_revenue"].sum().reset_index()

    fig = px.line(
        revenue_df,
        x=agg_column,
        y="total_revenue",
        title=f"📈 Total Revenue Over Time ({granularity}, {'All' if weekend_status is None else 'Weekends' if weekend_status else 'Weekdays'})",
        labels={"total_revenue": "Total Revenue ($)", agg_column: granularity.capitalize()},
        line_shape="spline",
        markers=True
    )
    fig.show()

for granularity in time_granularity_options:
    for label, weekend_status in weekend_filter_options:
        generate_plot(granularity, weekend_status)

### Question 8: What is the average fare per trip, and how does it vary by borough, time of day, and trip distance?


In [ ]:
query_avg_fare_analysis = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.avg_fare_analysis`
"""
avg_fare_analysis_df = query_to_dataframe(query_avg_fare_analysis)
avg_fare_analysis_df.head()

In [ ]:
# Filter rows where the year is between 2020 and the current year (inclusive)
filtered_avg_fare_analysis_df = avg_fare_analysis_df[(avg_fare_analysis_df['year'] >= 2020) & (avg_fare_analysis_df['year'] <= current_year)]
filtered_avg_fare_analysis_df.info()

In [ ]:
# 1. Heatmap: Analyze average fare by pickup and dropoff borough
fig_heatmap = px.density_heatmap(
    filtered_avg_fare_analysis_df,
    x="pickup_borough",
    y="dropoff_borough",
    z="avg_fare_per_trip",
    color_continuous_scale="Viridis",
    title="Heatmap of Average Fare per Trip by Pickup and Dropoff Borough",
    labels={"pickup_borough": "Pickup Borough", "dropoff_borough": "Dropoff Borough", "avg_fare_per_trip": "Average Fare ($)"},
)
fig_heatmap.show()

In [ ]:
# 2. Bar Chart: Average fare per trip by hour of the day
hourly_avg_fare = filtered_avg_fare_analysis_df.groupby("pickup_hour")["avg_fare_per_trip"].mean().reset_index()
fig_bar = px.bar(
    hourly_avg_fare,
    x="pickup_hour",
    y="avg_fare_per_trip",
    title="Average Fare per Trip by Hour of the Day",
    labels={"pickup_hour": "Pickup Hour", "avg_fare_per_trip": "Average Fare ($)"},
    text_auto=True,
)
fig_bar.update_traces(marker_color="blue", textfont_size=12)
fig_bar.show()

In [ ]:
# 3. Scatter Plot: Average fare vs. average trip distance
fig_scatter = px.scatter(
    filtered_avg_fare_analysis_df,
    x="avg_trip_distance",
    y="avg_fare_per_trip",
    size="total_trips",
    color="pickup_borough",
    title="Average Fare vs. Average Trip Distance (by Pickup Borough)",
    labels={"avg_trip_distance": "Average Trip Distance (miles)", "avg_fare_per_trip": "Average Fare ($)", "total_trips": "Total Trips"},
    hover_data=["pickup_borough", "dropoff_borough"],
)
fig_scatter.show()

### Question 9: What is the proportion of different payment types (credit card, cash, etc.), and has it changed over time?


In [ ]:
query_payment_type_trends = """
SELECT *
FROM `smiling-rhythm-486213-b9.views_fordashboard.payment_type_trends`
"""
payment_type_trends_df = query_to_dataframe(query_payment_type_trends)
payment_type_trends_df.head()

In [ ]:
# Filter rows where the year is between 2020 and the current year (inclusive)
filtered_payment_type_trends_df = payment_type_trends_df[(payment_type_trends_df['year'] >= 2020) & (payment_type_trends_df['year'] <= current_year)]
filtered_payment_type_trends_df.info()

In [ ]:
# Ensure 'trip_date' is in datetime format if not already
filtered_payment_type_trends_df["trip_date"] = pd.to_datetime(filtered_payment_type_trends_df["trip_date"])

In [ ]:
payment_summary = filtered_payment_type_trends_df.groupby("payment_method")["total_trips"].sum().reset_index()
payment_summary["percentage"] = (
    payment_summary["total_trips"] / payment_summary["total_trips"].sum()
) * 100

fig_donut = px.pie(
    payment_summary,
    names="payment_method",
    values="percentage",
    title="Proportion of Payment Methods",
    hole=0.5,
    labels={"payment_method": "Payment Method", "percentage": "Percentage (%)"},
)

fig_donut.update_traces(
    textinfo="percent+label",
    hoverinfo="label+percent+value",
    pull=[0.1 if p == payment_summary["percentage"].max() else 0 for p in payment_summary["percentage"]],
)

fig_donut.update_layout(
    showlegend=True,
    legend_title_text="Payment Methods",
    template="plotly_white",
)

fig_donut.show()

In [ ]:
fig = px.area(
    filtered_payment_type_trends_df,
    x="trip_date",
    y="payment_percentage",
    color="payment_method",
    title="Proportion of Different Payment Methods Over Time",
    labels={"payment_percentage": "Payment Share (%)", "trip_date": "Date"},
    template="plotly_white"
)

fig.show()